In [ ]:
!pip install -q --upgrade \
  "huggingface_hub>=0.26.2" "transformers>=4.46.0" "accelerate>=0.33.0" \
  "peft>=0.13.0" "sentence-transformers>=3.0.0" "faiss-cpu>=1.7.4" \
  "codecarbon>=2.3.0" "seaborn>=0.13.0" "psutil>=5.9.0"

In [ ]:
!pip install -q --upgrade "torchao>=0.16.0" "peft>=0.14.0"

In [ ]:
import importlib
import inspect
import shutil
import subprocess
import sys

def _version_tuple(version_str):
    parts = []
    for piece in str(version_str).split('.'):
        try:
            parts.append(int(piece))
        except ValueError:
            break
    return tuple(parts)


def ensure_torchao():
    """PEFT on recent Colab requires torchao>=0.16.0."""
    try:
        import torchao
        if _version_tuple(torchao.__version__) < (0, 16, 0):
            raise ImportError(f'torchao {torchao.__version__} is too old')
        return
    except ImportError:
        pass
    print('Upgrading torchao for PEFT / LoRA compatibility...')
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'torchao>=0.16.0',
    ])


def ensure_hf_stack():
    """
    transformers>=4.44 requires huggingface_hub with split_torch_state_dict_into_shards.
    On Colab, upgrade once per session; if imports still fail, restart runtime and re-run.
    """
    try:
        from huggingface_hub import split_torch_state_dict_into_shards  # noqa: F401
        ensure_torchao()
        return
    except ImportError:
        pass

    print('Upgrading huggingface_hub / transformers (fixes ImportError on Colab)...')
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
        'huggingface_hub>=0.26.2,<1.0',
        'transformers>=4.46.0,<5.0',
        'accelerate>=0.33.0,<2.0',
        'safetensors>=0.4.0',
        'tokenizers>=0.20.0',
        'peft>=0.14.0',
        'sentence-transformers>=3.0.0,<4.0',
        'codecarbon>=2.3.0,<3.0',
        'scikit-learn>=1.3.0',
        'seaborn>=0.13.0',
        'psutil>=5.9.0',
        'faiss-cpu>=1.7.4',
        'torchao>=0.16.0',
    ])
    import huggingface_hub
    importlib.reload(huggingface_hub)

    try:
        from huggingface_hub import split_torch_state_dict_into_shards  # noqa: F401
    except ImportError:
        raise RuntimeError(
            'huggingface_hub was upgraded but the import still fails. '
            'In Colab: Runtime → Restart session, then run this cell again.'
        ) from None
    ensure_torchao()


ensure_hf_stack()

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

import json
import os
import random
import time
import threading
import traceback

import numpy as np
import pandas as pd
import torch
import psutil
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from sklearn.model_selection import train_test_split
import faiss

import huggingface_hub
import codecarbon
from codecarbon import EmissionsTracker

from transformers import (
    AutoModelForQuestionAnswering,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    default_data_collator,
    TrainerCallback,
)
from peft import LoraConfig, get_peft_model, TaskType
from sentence_transformers import SentenceTransformer

ensure_torchao()

# =============================================================================
# Reproducibility
# =============================================================================
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# =============================================================================
# Experiment configuration
# =============================================================================
NUM_EPOCHS       = 3
BATCH_SIZE       = 8
SAMPLE_FRACTION  = 0.50
MAX_LENGTH       = 384
EVAL_SAMPLE_SIZE = 50
RAG_TOP_K        = 3   # FAISS retrieves top-k chunks; reader uses rank-1

CHUNK_CONFIG = {
    'chunk_size'   : 800,
    'chunk_overlap': 200,
    'strategy'     : 'sliding_window_character',
}

CC_CONFIG = {
    'country_iso_code'  : 'IND',
    'tracking_mode'     : 'machine',
    'log_level'         : 'warning',
    'save_to_file'      : True,
    'output_dir'        : 'emissions_data',
    'measure_power_secs': 10,
}

os.makedirs('emissions_data', exist_ok=True)
os.makedirs('results_comparison', exist_ok=True)

print('\n' + '=' * 65)
print('EXPERIMENT CONFIGURATION  (paper section 4.2)')
print('=' * 65)
print(f'  huggingface_hub      : {huggingface_hub.__version__}')
print(f'  transformers         : {__import__("transformers").__version__}')
print(f'  codecarbon version   : {codecarbon.__version__}')
print(f'  country_iso_code     : {CC_CONFIG["country_iso_code"]}')
print(f'  carbon_intensity     : 708 gCO2eq/kWh  (India CEA 2023)')
print(f'  tracking_mode        : {CC_CONFIG["tracking_mode"]}')
print(f'  device               : {DEVICE}')
if torch.cuda.is_available():
    print(f'  GPU                  : {torch.cuda.get_device_name(0)}')
print(f'  SAMPLE_FRACTION      : {SAMPLE_FRACTION}')
print(f'  NUM_EPOCHS           : {NUM_EPOCHS}')
print(f'  BATCH_SIZE           : {BATCH_SIZE}')
print(f'  MAX_LENGTH           : {MAX_LENGTH}')
print(f'  EVAL_SAMPLE_SIZE     : {EVAL_SAMPLE_SIZE}')
print(f'  random_seed          : {RANDOM_SEED}')
print(f'  chunk_size           : {CHUNK_CONFIG["chunk_size"]} chars')
print(f'  chunk_overlap        : {CHUNK_CONFIG["chunk_overlap"]} chars')
print(f'  RAG_retriever        : sentence-transformers/all-MiniLM-L6-v2')
print(f'  RAG_reader           : distilbert-base-cased-distilled-squad')
print(f'  RAG_knowledge_base   : AdvQA training split (same-dataset)')
print(f'  RAG_top_k            : {RAG_TOP_K}')
print(f'  RAG_search           : FAISS IndexFlatIP (cosine, L2-normalized)')
print('=' * 65 + '\n')


def _answer_token_span(offset_mapping, sequence_ids, answer_start, answer_end):
    """Map character answer span to inclusive token start/end in the context (seq_id==1)."""
    token_start = 0
    token_end = 0
    found_start = False

    for i, (offset, seq_id) in enumerate(zip(offset_mapping, sequence_ids)):
        if seq_id != 1:
            continue
        if offset[0] is None or offset[1] is None:
            continue
        if offset[0] <= answer_start < offset[1]:
            token_start = i
            found_start = True
        if offset[0] < answer_end <= offset[1]:
            token_end = i

    if found_start and token_end < token_start:
        token_end = token_start
    return token_start, token_end


def build_faiss_index(embeddings):
    """
    Build a FAISS IndexFlatIP over L2-normalized chunk embeddings.
    Inner product on unit vectors equals cosine similarity.
    """
    vectors = np.ascontiguousarray(np.asarray(embeddings, dtype=np.float32))
    if vectors.ndim != 2 or vectors.shape[0] == 0:
        raise ValueError('embeddings must be a non-empty 2-D array')
    faiss.normalize_L2(vectors)
    dim = vectors.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(vectors)
    return index


def faiss_search(index, query_embeddings, top_k=RAG_TOP_K):
    """
    Search the FAISS index. Returns (scores, indices) with shape (n_queries, top_k).
    Scores are cosine similarities when vectors are L2-normalized.
    """
    queries = np.ascontiguousarray(np.asarray(query_embeddings, dtype=np.float32))
    if queries.ndim == 1:
        queries = queries.reshape(1, -1)
    faiss.normalize_L2(queries)
    k = min(int(top_k), index.ntotal)
    scores, indices = index.search(queries, k)
    return scores, indices


def make_training_arguments(model_type):
    """
    Build TrainingArguments compatible with transformers 4.x and 5.x.
    v5 removed overwrite_output_dir and logging_dir — output dir is cleared manually.
    """
    out_dir = f'./results_comparison/{model_type}'
    if os.path.isdir(out_dir):
        shutil.rmtree(out_dir, ignore_errors=True)
    os.makedirs(out_dir, exist_ok=True)

    desired = {
        'output_dir'                  : out_dir,
        'overwrite_output_dir'        : True,   # transformers < 5 only
        'num_train_epochs'            : NUM_EPOCHS,
        'per_device_train_batch_size' : BATCH_SIZE,
        'per_device_eval_batch_size'  : BATCH_SIZE,
        'learning_rate'               : 5e-5,
        'weight_decay'                : 0.01,
        'warmup_steps'                : 100,
        'logging_dir'                 : './logs',  # transformers < 5 only
        'logging_steps'               : 50,
        'save_steps'                  : 200,
        'save_total_limit'            : 2,
        'report_to'                   : 'none',
        'disable_tqdm'                : False,
    }
    if not torch.cuda.is_available():
        desired['use_cpu'] = True   # transformers 5.x
        desired['no_cuda'] = True   # transformers 4.x

    sig = inspect.signature(TrainingArguments.__init__)
    valid = set(sig.parameters) - {'self'}
    ta_kw = {k: v for k, v in desired.items() if k in valid}

    dropped = sorted(set(desired) - set(ta_kw))
    if dropped:
        print(f'  TrainingArguments: skipped unsupported keys: {dropped}')

    return TrainingArguments(**ta_kw)


def safe_emissions_tracker(output_file, **kwargs):
    """Create an EmissionsTracker compatible with CodeCarbon 2.x and 3.x."""
    save_to_file = kwargs.pop('save_to_file', CC_CONFIG['save_to_file'])
    measure_power_secs = kwargs.pop('measure_power_secs', CC_CONFIG['measure_power_secs'])
    common = dict(
        output_file=output_file,
        tracking_mode=CC_CONFIG['tracking_mode'],
        log_level=CC_CONFIG['log_level'],
        save_to_file=save_to_file,
        output_dir=CC_CONFIG['output_dir'],
        measure_power_secs=measure_power_secs,
        **kwargs,
    )
    try:
        return EmissionsTracker(
            country_iso_code=CC_CONFIG['country_iso_code'],
            **common,
        )
    except TypeError:
        return EmissionsTracker(**common)


class TextChunker:
    """
    Sliding-window character-based chunker for RAG indexing.

    Applied ONLY to RAG context indexing.
    Fine-tuning and LoRA use full, unchunked contexts.

    Paper (Section 3.3):
      chunk_size    = 800 chars  (~200 tokens at 4 chars/token)
      chunk_overlap = 200 chars  (~50 tokens)
      Ensures chunk + question (~30 tokens) fits within MAX_LENGTH=512.
    """

    def __init__(
        self,
        chunk_size    = CHUNK_CONFIG['chunk_size'],
        chunk_overlap = CHUNK_CONFIG['chunk_overlap'],
    ):
        assert chunk_overlap < chunk_size, \
            "chunk_overlap must be smaller than chunk_size"
        self.chunk_size    = chunk_size
        self.chunk_overlap = chunk_overlap
        self.step          = chunk_size - chunk_overlap

    def chunk_text(self, text, source_idx=None):
        chunks = []
        text   = text.strip()
        start  = 0

        while start < len(text):
            end   = min(start + self.chunk_size, len(text))
            chunk = text[start:end].strip()

            if len(chunk) > 30:
                chunks.append({
                    'text'      : chunk,
                    'char_start': start,
                    'char_end'  : end,
                    'source_idx': source_idx,
                    'chunk_id'  : len(chunks),
                })

            if end == len(text):
                break
            start += self.step

        return chunks

    def chunk_dataset(self, train_data):
        all_chunks = []
        for idx, example in enumerate(train_data):
            chunks = self.chunk_text(example['context'], source_idx=idx)
            for chunk in chunks:
                chunk['gold_answer'] = example['answers']['text'][0]
                chunk['question']    = example['question']
            all_chunks.extend(chunks)
        return all_chunks

    def print_stats(self, all_chunks, train_data):
        lens = [len(c['text']) for c in all_chunks]
        print('\n  Chunking statistics (paper section 3.3):')
        print(f'    original contexts  : {len(train_data)}')
        print(f'    total chunks       : {len(all_chunks)}')
        print(f'    avg chunks/context : {len(all_chunks)/len(train_data):.1f}')
        print(f'    avg chunk length   : {np.mean(lens):.0f} chars')
        print(f'    min / max length   : {min(lens)} / {max(lens)} chars')
        print(f'    chunk_size         : {self.chunk_size} chars')
        print(f'    chunk_overlap      : {self.chunk_overlap} chars')
        print(f'    step               : {self.step} chars')


class QADataset(torch.utils.data.Dataset):
    """
    Dataset for extractive QA (DistilBERT full fine-tune or LoRA).

    start_positions and end_positions are derived from offset_mapping,
    mapping character-level answer_start correctly to token positions.
    """

    def __init__(self, tokenizer, data):
        self.tokenizer  = tokenizer
        self.data       = data
        self.max_length = MAX_LENGTH

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        try:
            item         = self.data[idx]
            answer_start = item['answers']['answer_start'][0]
            answer_text  = item['answers']['text'][0]
            answer_end   = answer_start + len(answer_text)

            encoding = self.tokenizer(
                item['question'],
                item['context'],
                truncation             = True,
                padding                = 'max_length',
                max_length             = self.max_length,
                return_tensors         = 'pt',
                return_offsets_mapping = True,
            )

            offset_mapping = encoding['offset_mapping'][0].tolist()
            sequence_ids   = encoding.encodings[0].sequence_ids
            token_start, token_end = _answer_token_span(
                offset_mapping, sequence_ids, answer_start, answer_end
            )

            encoding.pop('offset_mapping')

            return {
                'input_ids'      : encoding['input_ids'].squeeze(0),
                'attention_mask' : encoding['attention_mask'].squeeze(0),
                'start_positions': torch.tensor(token_start, dtype=torch.long),
                'end_positions'  : torch.tensor(token_end,   dtype=torch.long),
            }

        except Exception as e:
            print(f"QADataset error at idx {idx}: {e}")
            dummy = self.tokenizer(
                "dummy question", "dummy context",
                truncation=True, padding='max_length',
                max_length=self.max_length, return_tensors='pt',
            )
            return {
                'input_ids'      : dummy['input_ids'].squeeze(0),
                'attention_mask' : dummy['attention_mask'].squeeze(0),
                'start_positions': torch.tensor(0, dtype=torch.long),
                'end_positions'  : torch.tensor(1, dtype=torch.long),
            }


class CustomCallback(TrainerCallback):
    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step % args.logging_steps == 0:
            loss = (state.log_history[-1].get('loss', 'N/A')
                    if state.log_history else 'N/A')
            print(f"  Step {state.global_step}/{state.max_steps} — loss: {loss}")


class QASystem:
    """
    Runs one adaptation experiment and returns a comparable results dict.

    model_type: 'distilbert' | 'distilbert-lora' | 'distilbert-rag'
    """

    def __init__(self, model_type='distilbert'):
        self.model_type = model_type

    def _read_power(self):
        pw = {'cpu_power': 0.0, 'ram_power': 0.0, 'gpu_power': 0.0}
        try:
            pct  = psutil.cpu_percent(interval=0.1)
            freq = (psutil.cpu_freq().current / 1000
                    if psutil.cpu_freq() else 2.5)
            pw['cpu_power'] = (pct / 100) * psutil.cpu_count() * (15 + 10 * (freq - 2.5))
            pw['ram_power'] = (psutil.virtual_memory().percent / 100) * 5.0
            if torch.cuda.is_available():
                try:
                    pw['gpu_power'] = (
                        torch.cuda.power_draw(torch.cuda.current_device()) / 1000
                    )
                except Exception:
                    try:
                        util = torch.cuda.utilization(torch.cuda.current_device())
                        pw['gpu_power'] = min(50 + util * 100, 150)
                    except Exception:
                        pw['gpu_power'] = 75.0
        except Exception as e:
            print(f"Power read error: {e}")
        return pw

    def _power_thread(self, readings, stop_evt):
        while not stop_evt.is_set():
            readings.append(self._read_power())
            time.sleep(1)

    def _summarise_power(self, readings, prefix):
        if not readings:
            return {f'{prefix}_cpu': 0, f'{prefix}_ram': 0,
                    f'{prefix}_gpu': 0, f'{prefix}_total': 0}
        df = pd.DataFrame(readings)
        return {
            f'{prefix}_cpu'  : float(df['cpu_power'].mean()),
            f'{prefix}_ram'  : float(df['ram_power'].mean()),
            f'{prefix}_gpu'  : float(df['gpu_power'].mean()),
            f'{prefix}_total': float(
                df[['cpu_power', 'ram_power', 'gpu_power']].sum(axis=1).mean()
            ),
        }

    def load_and_sample_data(self, file_path, fraction=SAMPLE_FRACTION):
        print(f"\nLoading {fraction*100:.0f}% sample from {file_path}...")

        if not os.path.exists(file_path):
            raise FileNotFoundError(f"Dataset not found: {file_path}")

        with open(file_path, 'r') as f:
            first = f.read(1); f.seek(0)
            raw = (json.load(f) if first == '['
                   else [json.loads(l) for l in f if l.strip()])

        valid = []
        for ex in raw:
            try:
                if not all(k in ex for k in ['context', 'question', 'answers']):
                    continue
                ans = ex['answers']
                if not (isinstance(ans, dict)
                        and all(k in ans for k in ['text', 'answer_start'])
                        and ans['text'] and ans['answer_start']):
                    continue
                ctx = ex['context']
                a_s = ans['answer_start'][0]
                a_t = ans['text'][0]
                a_e = a_s + len(a_t)
                if not (0 <= a_s < len(ctx) and a_e <= len(ctx)
                        and ctx[a_s:a_e] == a_t):
                    continue
                valid.append(ex)
            except Exception:
                continue

        if not valid:
            raise ValueError("No valid QA pairs found in dataset.")

        n       = min(int(len(valid) * fraction), len(valid))
        sampled = random.sample(valid, n)

        train_data, val_data = train_test_split(
            sampled, test_size=0.2, random_state=RANDOM_SEED
        )
        print(f"✓ {len(valid)} valid → {n} sampled → "
              f"{len(train_data)} train / {len(val_data)} val "
              f"(80/20, seed={RANDOM_SEED}, no stratification)")
        return train_data, val_data

    def initialize_model(self):
        print(f"\nLoading {self.model_type}...")
        try:
            if self.model_type == 'distilbert':
                name = 'distilbert-base-cased-distilled-squad'
                tok  = AutoTokenizer.from_pretrained(name)
                mdl  = AutoModelForQuestionAnswering.from_pretrained(name).to(DEVICE)

            elif self.model_type == 'distilbert-lora':
                ensure_torchao()
                name = 'distilbert-base-cased-distilled-squad'
                tok  = AutoTokenizer.from_pretrained(name)
                mdl  = AutoModelForQuestionAnswering.from_pretrained(name).to(DEVICE)
                cfg  = LoraConfig(
                    task_type      = TaskType.QUESTION_ANS,
                    inference_mode = False,
                    r              = 8,
                    lora_alpha     = 32,
                    lora_dropout   = 0.1,
                    target_modules = ['q_lin', 'v_lin'],
                )
                mdl = get_peft_model(mdl, cfg)
                mdl.print_trainable_parameters()

            elif self.model_type == 'distilbert-rag':
                name = 'distilbert-base-cased-distilled-squad'
                tok  = AutoTokenizer.from_pretrained(name)
                mdl  = AutoModelForQuestionAnswering.from_pretrained(name).to(DEVICE)

            else:
                raise ValueError(f"Unknown model_type: {self.model_type}")

            print(f"✓ {self.model_type} loaded on {next(mdl.parameters()).device}")
            return {'model': mdl, 'tokenizer': tok}

        except Exception as e:
            print(f"❌ Model loading failed: {e}")
            if 'torchao' in str(e).lower():
                print("Fix: pip install -q 'torchao>=0.16.0' then restart runtime.")
            if 'CUDA out of memory' in str(e):
                print("Try reducing BATCH_SIZE or MAX_LENGTH.")
            return None

    def train_distilbert_model(self, model_dict, train_data, val_data):
        """
        Gradient-based training (full fine-tune or LoRA).
        EmissionsTracker wraps the entire trainer.train() call.
        """
        train_ds = QADataset(model_dict['tokenizer'], train_data)
        val_ds   = QADataset(model_dict['tokenizer'], val_data)

        args = make_training_arguments(self.model_type)

        trainer = Trainer(
            model         = model_dict['model'],
            args          = args,
            train_dataset = train_ds,
            eval_dataset  = val_ds,
            data_collator = default_data_collator,
            callbacks     = [CustomCallback()],
        )

        readings = []
        stop_evt = threading.Event()
        p_thread = threading.Thread(
            target=self._power_thread, args=(readings, stop_evt), daemon=True
        )
        tracker = safe_emissions_tracker(
            output_file=f'emissions_{self.model_type}_train.csv'
        )

        print(f"\n🔧 Training {self.model_type}...")
        emissions = 0.0
        try:
            p_thread.start()
            tracker.start()
            trainer.train()
            emissions = tracker.stop()
        except Exception:
            try:
                tracker.stop()
            except Exception:
                pass
            raise
        finally:
            stop_evt.set()
            p_thread.join()

        print(f"✓ Training emissions: {emissions:.8f} kg CO₂")
        return model_dict, emissions, self._summarise_power(readings, 'train')

    def train_rag_model(self, model_dict, train_data, val_data):
        """
        RAG Phase 1 — offline chunking + embedding indexing.

        Retriever : all-MiniLM-L6-v2
        Reader    : distilbert-base-cased-distilled-squad (no fine-tune)
        KB        : AdvQA training split (same-dataset)
        Search    : FAISS IndexFlatIP (cosine via L2-normalized inner product)
        Emissions : real CodeCarbon measurement (no mock sleep).
        """
        print('\n📚 RAG — offline indexing (Phase 1)...')
        print(f'   retriever : all-MiniLM-L6-v2')
        print(f'   reader    : distilbert-base-cased-distilled-squad')
        print(f'   KB source : AdvQA training split (same-dataset)')
        print(f'   chunk     : size={CHUNK_CONFIG["chunk_size"]} chars, '
              f'overlap={CHUNK_CONFIG["chunk_overlap"]} chars')

        retriever = SentenceTransformer('all-MiniLM-L6-v2')
        chunker   = TextChunker()

        readings = []
        stop_evt = threading.Event()
        p_thread = threading.Thread(
            target=self._power_thread, args=(readings, stop_evt), daemon=True
        )
        tracker = safe_emissions_tracker(
            output_file='emissions_distilbert-rag_indexing.csv'
        )

        t_start = time.time()
        all_chunks = []
        emissions = 0.0

        try:
            p_thread.start()
            tracker.start()

            all_chunks = chunker.chunk_dataset(train_data)
            chunker.print_stats(all_chunks, train_data)

            chunk_texts    = [c['text'] for c in all_chunks]
            ENCODE_BATCH   = 64
            all_embeddings = []

            print(f'\n   Encoding {len(all_chunks)} chunks...')
            for i in tqdm(range(0, len(chunk_texts), ENCODE_BATCH),
                          desc='   Encoding'):
                batch = chunk_texts[i: i + ENCODE_BATCH]
                embs  = retriever.encode(
                    batch, convert_to_numpy=True, show_progress_bar=False
                )
                all_embeddings.append(embs)

            all_embeddings = np.vstack(all_embeddings).astype(np.float32)

            for i, chunk in enumerate(all_chunks):
                chunk['embedding'] = all_embeddings[i]

            emissions = tracker.stop()

        except Exception:
            try:
                tracker.stop()
            except Exception:
                pass
            raise
        finally:
            stop_evt.set()
            p_thread.join()

        elapsed = time.time() - t_start

        pd.DataFrame([{
            'num_source_contexts'  : len(train_data),
            'num_chunks'           : len(all_chunks),
            'chunk_size_chars'     : CHUNK_CONFIG['chunk_size'],
            'chunk_overlap_chars'  : CHUNK_CONFIG['chunk_overlap'],
            'avg_chunks_per_ctx'   : round(len(all_chunks) / len(train_data), 2),
            'embedding_model'      : 'all-MiniLM-L6-v2',
            'embedding_dim'        : 384,
            'storage'              : 'FAISS IndexFlatIP (in-memory)',
            'search_method'        : 'FAISS inner product (L2-normalized cosine)',
            'faiss_index_type'     : 'IndexFlatIP',
            'top_k'                : RAG_TOP_K,
            'indexing_time_sec'    : round(elapsed, 2),
            'indexing_emissions_kg': emissions,
            'country_iso_code'     : CC_CONFIG['country_iso_code'],
            'carbon_intensity'     : '708 gCO2eq/kWh',
        }]).to_csv('emissions_data/rag_indexing_metadata.csv', index=False)

        print(f'\n✓ Indexing: {emissions:.8f} kg CO₂  in {elapsed:.1f}s')
        print(f'   Building FAISS index ({all_embeddings.shape[0]} vectors, dim={all_embeddings.shape[1]})...')
        faiss_index = build_faiss_index(all_embeddings)

        model_dict['retriever']      = retriever
        model_dict['all_chunks']     = all_chunks
        model_dict['all_embeddings'] = all_embeddings
        model_dict['faiss_index']    = faiss_index

        return model_dict, emissions, self._summarise_power(readings, 'train')

    def _retrieve_chunk(self, model_dict, question, top_k=RAG_TOP_K):
        """Encode question and retrieve the best matching chunk via FAISS."""
        q_emb = model_dict['retriever'].encode(
            question,
            convert_to_numpy=True,
            show_progress_bar=False,
        )
        scores, indices = faiss_search(model_dict['faiss_index'], q_emb, top_k=top_k)
        best_idx = int(indices[0][0])
        best_sim = float(scores[0][0])
        context  = model_dict['all_chunks'][best_idx]['text']
        return best_idx, best_sim, context

    def _predict_span(self, model, tok, question, context):
        enc = tok(
            question,
            context,
            truncation             = True,
            max_length             = MAX_LENGTH,
            return_tensors         = 'pt',
            padding                = True,
            return_offsets_mapping = True,
        )
        offset_mapping = enc.pop('offset_mapping')[0].tolist()
        sequence_ids   = enc.encodings[0].sequence_ids
        enc            = {k: v.to(DEVICE) for k, v in enc.items()}

        with torch.no_grad():
            out          = model(**enc)
            start_logits = out.start_logits[0]
            end_logits   = out.end_logits[0]

        start_l = [
            start_logits[i].item() if sequence_ids[i] == 1
            else float('-inf')
            for i in range(len(sequence_ids))
        ]
        end_l = [
            end_logits[i].item() if sequence_ids[i] == 1
            else float('-inf')
            for i in range(len(sequence_ids))
        ]

        s_idx = int(np.argmax(start_l))
        e_idx = int(np.argmax(end_l))
        if e_idx < s_idx:
            e_idx = s_idx

        s_char = offset_mapping[s_idx][0]
        e_char = offset_mapping[e_idx][1]
        if s_char is None:
            s_char = 0
        if e_char is None:
            e_char = len(context)

        pred_answer = context[s_char:e_char].strip()
        return pred_answer

    def evaluate_model(self, model_dict, val_data):
        """
        Evaluate on val_data[:EVAL_SAMPLE_SIZE].
        Span extraction via offset_mapping; per-example CodeCarbon tracking.
        RAG: query embedding + cosine search + reader inference per call.
        """
        eval_emissions = []
        f1_scores      = []
        exact_matches  = []
        power_data     = []
        retrieval_hits = []

        model = model_dict['model']
        tok   = model_dict['tokenizer']
        model.eval()

        n_eval = min(len(val_data), EVAL_SAMPLE_SIZE)
        print(f"\n🔍 Evaluating {self.model_type} on {n_eval} examples...")

        for idx, qa_pair in enumerate(tqdm(val_data[:n_eval], desc='Eval')):
            readings = []
            stop_evt = threading.Event()
            p_thread = threading.Thread(
                target=self._power_thread, args=(readings, stop_evt), daemon=True
            )
            tracker = safe_emissions_tracker(
                output_file=f'emissions_{self.model_type}_eval.csv',
                save_to_file=False,
                measure_power_secs=1,
            )

            p_thread.start()
            emissions = 0.0
            try:
                tracker.start()

                if self.model_type == 'distilbert-rag':
                    best_idx, best_sim, context = self._retrieve_chunk(
                        model_dict, qa_pair['question']
                    )

                    gold_ans     = qa_pair['answers']['text'][0]
                    ans_in_chunk = gold_ans.lower() in context.lower()
                    retrieval_hits.append(int(ans_in_chunk))

                    if idx < 5:
                        print(f"\n  [RAG debug #{idx}]")
                        print(f"    Q            : {qa_pair['question'][:65]}...")
                        print(f"    Similarity   : {best_sim:.4f}")
                        print(f"    Ans in chunk : {ans_in_chunk}")
                        print(f"    Chunk[:60]   : {context[:60]}...")
                else:
                    context = qa_pair['context']

                pred_answer = self._predict_span(
                    model, tok, qa_pair['question'], context
                )

                em, f1 = self.calculate_metrics(
                    pred_answer, qa_pair['answers']['text']
                )
                exact_matches.append(em)
                f1_scores.append(f1)

            except Exception as e:
                print(f"\n⚠️ Eval error #{idx}: {str(e)[:120]}")
                exact_matches.append(0)
                f1_scores.append(0)
                if self.model_type == 'distilbert-rag':
                    retrieval_hits.append(0)

            finally:
                try:
                    emissions = tracker.stop()
                except Exception:
                    emissions = 0.0
                stop_evt.set()
                p_thread.join()
                eval_emissions.append(emissions)
                power_data.append(self._summarise_power(readings, 'eval'))

        if retrieval_hits:
            retr_acc = float(np.mean(retrieval_hits))
            print(f'\n📊 Retrieval accuracy (answer in chunk): {retr_acc*100:.1f}%')
            pd.DataFrame([{
                'retrieval_accuracy'   : retr_acc,
                'n_correct_retrieval'  : sum(retrieval_hits),
                'n_total'              : len(retrieval_hits),
                'chunk_size_chars'     : CHUNK_CONFIG['chunk_size'],
                'chunk_overlap_chars'  : CHUNK_CONFIG['chunk_overlap'],
                'top_k'                : RAG_TOP_K,
                'search_backend'       : 'FAISS IndexFlatIP',
                'embedder'             : 'all-MiniLM-L6-v2',
            }]).to_csv('emissions_data/rag_retrieval_diagnostic.csv', index=False)

        avg_ep = {
            'cpu'  : float(np.mean([p['eval_cpu']   for p in power_data])),
            'ram'  : float(np.mean([p['eval_ram']   for p in power_data])),
            'gpu'  : float(np.mean([p['eval_gpu']   for p in power_data])),
            'total': float(np.mean([p['eval_total'] for p in power_data])),
        }

        return eval_emissions, f1_scores, exact_matches, avg_ep

    @staticmethod
    def calculate_metrics(pred, true_answers):
        def norm(t):
            t = str(t).lower()
            t = ''.join(c for c in t if c.isalnum() or c.isspace())
            return ' '.join(t.split())

        p  = norm(pred)
        ts = [norm(a) for a in true_answers]
        em = int(any(p == t for t in ts))

        p_tok = p.split()
        best  = 0.0
        for t in ts:
            t_tok  = t.split()
            if not p_tok or not t_tok:
                continue
            common = set(p_tok) & set(t_tok)
            prec   = len(common) / len(p_tok)
            rec    = len(common) / len(t_tok)
            f1     = (2 * prec * rec / (prec + rec)) if (prec + rec) > 0 else 0.0
            best   = max(best, f1)

        return em, best

    def diagnose_rag_performance(self, model_dict, val_data):
        """
        Conditional F1/EM given correct vs wrong retrieval (paper section 5.3).
        """
        if self.model_type != 'distilbert-rag':
            return

        print('\n🔬 RAG performance diagnostic (paper section 5.3)...')
        model = model_dict['model']
        tok   = model_dict['tokenizer']
        model.eval()

        correct_f1 = []
        wrong_f1   = []
        correct_em = []
        wrong_em   = []
        hits       = []

        for qa in tqdm(val_data[:EVAL_SAMPLE_SIZE], desc='  Diagnostic'):
            try:
                _, _, context = self._retrieve_chunk(model_dict, qa['question'])
                gold     = qa['answers']['text'][0]
                hit      = int(gold.lower() in context.lower())
                hits.append(hit)

                pred   = self._predict_span(model, tok, qa['question'], context)
                em, f1 = self.calculate_metrics(pred, qa['answers']['text'])

                if hit:
                    correct_f1.append(f1); correct_em.append(em)
                else:
                    wrong_f1.append(f1);   wrong_em.append(em)

            except Exception:
                hits.append(0)
                wrong_f1.append(0); wrong_em.append(0)

        diag = {
            'retrieval_accuracy'  : float(np.mean(hits)),
            'overall_f1'          : float(np.mean(correct_f1 + wrong_f1)) if hits else 0.0,
            'f1_given_correct_ctx': float(np.mean(correct_f1)) if correct_f1 else 0.0,
            'f1_given_wrong_ctx'  : float(np.mean(wrong_f1))   if wrong_f1   else 0.0,
            'overall_em'          : float(np.mean(correct_em + wrong_em)) if hits else 0.0,
            'em_given_correct_ctx': float(np.mean(correct_em)) if correct_em else 0.0,
            'em_given_wrong_ctx'  : float(np.mean(wrong_em))   if wrong_em   else 0.0,
            'n_correct_retrieval' : int(sum(hits)),
            'n_wrong_retrieval'   : int(len(hits) - sum(hits)),
        }

        print('\n  Diagnostic results:')
        for k, v in diag.items():
            if isinstance(v, float):
                print(f'    {k:<30} : {v:.4f}')
            else:
                print(f'    {k:<30} : {v}')

        pd.DataFrame([diag]).to_csv(
            'emissions_data/rag_performance_diagnostic.csv', index=False
        )
        return diag

    def run_experiment(self, data_path):
        print(f'\n{"="*60}')
        print(f'🚀 {self.model_type.upper()} experiment')
        print(f'   device={DEVICE}  epochs={NUM_EPOCHS}  batch={BATCH_SIZE}')

        train_data, val_data = self.load_and_sample_data(data_path)
        model_dict           = self.initialize_model()
        if not model_dict:
            return None

        if self.model_type == 'distilbert-rag':
            model_dict, t_em, t_pw = self.train_rag_model(
                model_dict, train_data, val_data
            )
            self.diagnose_rag_performance(model_dict, val_data)
        else:
            model_dict, t_em, t_pw = self.train_distilbert_model(
                model_dict, train_data, val_data
            )

        e_ems, f1s, ems, e_pw = self.evaluate_model(model_dict, val_data)

        results = {
            'Model'              : self.model_type,
            'Train_CO2_kg'       : t_em,
            'Train_phase'        : ('indexing'    if self.model_type == 'distilbert-rag'
                                    else 'fine-tuning'),
            'Eval_CO2_kg_mean'   : float(np.mean(e_ems)),
            'Eval_CO2_kg_std'    : float(np.std(e_ems)),
            'Avg_F1'             : float(np.mean(f1s)),
            'Std_F1'             : float(np.std(f1s)),
            'Avg_EM'             : float(np.mean(ems)),
            'Std_EM'             : float(np.std(ems)),
            'Train_CPU_W'        : t_pw['train_cpu'],
            'Train_RAM_W'        : t_pw['train_ram'],
            'Train_GPU_W'        : t_pw['train_gpu'],
            'Train_Total_W'      : t_pw['train_total'],
            'Eval_CPU_W'         : e_pw['cpu'],
            'Eval_RAM_W'         : e_pw['ram'],
            'Eval_GPU_W'         : e_pw['gpu'],
            'Eval_Total_W'       : e_pw['total'],
            'Num_Train'          : len(train_data),
            'Num_Eval'           : min(len(val_data), EVAL_SAMPLE_SIZE),
            'Batch_Size'         : BATCH_SIZE,
            'Epochs'             : NUM_EPOCHS,
            'Chunk_Size_Chars'   : (CHUNK_CONFIG['chunk_size']
                                    if self.model_type == 'distilbert-rag' else 'N/A'),
            'Chunk_Overlap_Chars': (CHUNK_CONFIG['chunk_overlap']
                                    if self.model_type == 'distilbert-rag' else 'N/A'),
        }

        print(f'\n🎯 {self.model_type} results:')
        print(pd.DataFrame([results])[
            ['Model', 'Train_CO2_kg', 'Eval_CO2_kg_mean', 'Avg_F1', 'Avg_EM']
        ].to_string(index=False))

        return results


def chunk_size_ablation(train_data, val_data, save=True):
    """Ablation over chunk_size × overlap using answer-in-chunk retrieval accuracy."""
    print('\n📐 Chunk-size ablation...')
    retriever = SentenceTransformer('all-MiniLM-L6-v2')

    configs = [
        {'chunk_size': 400,  'chunk_overlap': 100},
        {'chunk_size': 800,  'chunk_overlap': 200},
        {'chunk_size': 1200, 'chunk_overlap': 300},
    ]
    rows = []

    for cfg in configs:
        chunker    = TextChunker(cfg['chunk_size'], cfg['chunk_overlap'])
        all_chunks = chunker.chunk_dataset(train_data)
        texts      = [c['text'] for c in all_chunks]
        embs       = retriever.encode(
            texts, convert_to_numpy=True, batch_size=64, show_progress_bar=False
        ).astype(np.float32)
        faiss_index = build_faiss_index(embs)

        hits = []
        for qa in val_data[:EVAL_SAMPLE_SIZE]:
            q_emb = retriever.encode(
                [qa['question']], convert_to_numpy=True, show_progress_bar=False
            )
            _, indices = faiss_search(faiss_index, q_emb, top_k=1)
            best_idx = int(indices[0][0])
            chunk    = all_chunks[best_idx]['text']
            gold     = qa['answers']['text'][0]
            hits.append(int(gold.lower() in chunk.lower()))

        row = {
            'chunk_size'    : cfg['chunk_size'],
            'chunk_overlap' : cfg['chunk_overlap'],
            'num_chunks'    : len(all_chunks),
            'avg_chunks_ctx': round(len(all_chunks) / len(train_data), 1),
            'retrieval_acc' : round(float(np.mean(hits)), 4),
        }
        rows.append(row)
        print(f"  chunk={cfg['chunk_size']:5d}  overlap={cfg['chunk_overlap']:4d}  "
              f"chunks={len(all_chunks):5d}  acc={row['retrieval_acc']:.4f}")

    df = pd.DataFrame(rows)
    if save:
        df.to_csv('emissions_data/chunk_ablation.csv', index=False)
    print('\nAblation table:')
    print(df.to_string(index=False))
    return df


def break_even_analysis(results_df, save=True):
    """
    Break-even Q* where cumulative emissions of FT/LoRA cross RAG (paper 5.4).

    Q* = (E_train_a - E_index_rag) / (E_infer_rag - E_infer_a)
    """
    print('\n📈 Break-even analysis...')

    ft   = results_df[results_df['Model'] == 'distilbert']
    lora = results_df[results_df['Model'] == 'distilbert-lora']
    rag  = results_df[results_df['Model'] == 'distilbert-rag']

    if ft.empty or lora.empty or rag.empty:
        print('Need all 3 model results — skipping break-even analysis.')
        return None, None

    E_train_ft   = float(ft['Train_CO2_kg'].values[0])
    E_train_lora = float(lora['Train_CO2_kg'].values[0])
    E_index_rag  = float(rag['Train_CO2_kg'].values[0])
    E_infer_ft   = float(ft['Eval_CO2_kg_mean'].values[0])
    E_infer_lora = float(lora['Eval_CO2_kg_mean'].values[0])
    E_infer_rag  = float(rag['Eval_CO2_kg_mean'].values[0])

    Q      = np.arange(0, 10001, 100)
    E_ft   = (E_train_ft   + Q * E_infer_ft)   * 1000
    E_lora = (E_train_lora + Q * E_infer_lora) * 1000
    E_rag  = (E_index_rag  + Q * E_infer_rag)  * 1000

    def q_star(E_fa, E_va, E_fb, E_vb):
        d = E_vb - E_va
        if abs(d) < 1e-12:
            return None
        q = (E_fa - E_fb) / d
        return float(q) if q > 0 else None

    q_ft_rag   = q_star(E_train_ft,   E_infer_ft,   E_index_rag, E_infer_rag)
    q_lora_rag = q_star(E_train_lora, E_infer_lora, E_index_rag, E_infer_rag)

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(Q, E_ft,   label='Fine-tuning (DistilBERT)', color='steelblue',  lw=2)
    ax.plot(Q, E_lora, label='LoRA (DistilBERT)',         color='darkorange', lw=2)
    ax.plot(Q, E_rag,  label='RAG (DistilBERT)',           color='green',      lw=2, ls='--')

    beq = [
        (q_ft_rag,   'steelblue',  'FT–RAG'),
        (q_lora_rag, 'darkorange', 'LoRA–RAG'),
    ]
    for q, col, lbl in beq:
        if q and q <= 10000:
            ax.axvline(q, color=col, ls=':', alpha=0.7,
                       label=f'{lbl} Q*≈{int(q):,}')

    ax.set_xlabel('Number of queries (Q)', fontsize=12)
    ax.set_ylabel('Cumulative CO₂ (g CO₂eq)', fontsize=12)
    ax.set_title('Break-even analysis: cumulative emissions vs query volume',
                 fontsize=13)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()

    if save:
        plt.savefig('breakeven_analysis.png', dpi=300, bbox_inches='tight')
        print("  Saved: breakeven_analysis.png")
    if IN_COLAB:
        plt.show()
    else:
        plt.close(fig)

    print(f"  FT   vs RAG : Q* ≈ {int(q_ft_rag):,}"
          if q_ft_rag   else "  FT   vs RAG : RAG always higher cost")
    print(f"  LoRA vs RAG : Q* ≈ {int(q_lora_rag):,}"
          if q_lora_rag else "  LoRA vs RAG : RAG always higher cost")

    pd.DataFrame([{
        'E_train_ft_kg'  : E_train_ft,
        'E_train_lora_kg': E_train_lora,
        'E_index_rag_kg' : E_index_rag,
        'E_infer_ft_kg'  : E_infer_ft,
        'E_infer_lora_kg': E_infer_lora,
        'E_infer_rag_kg' : E_infer_rag,
        'Q_star_ft_rag'  : q_ft_rag,
        'Q_star_lora_rag': q_lora_rag,
    }]).to_csv('emissions_data/breakeven_results.csv', index=False)

    return q_ft_rag, q_lora_rag


def visualize_comparison(results, save=True):
    df = pd.DataFrame(results)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle('DistilBERT adaptation comparison', fontsize=14)

    specs = [
        ('Train_CO2_kg',    'Training CO₂ (kg)',  'Blues',   None),
        ('Avg_F1',          'Average F1',          'Greens',  1),
        ('Avg_EM',          'Average exact match', 'Oranges', 1),
    ]

    for ax, (col, title, pal, ylim) in zip(axes, specs):
        sns.barplot(x='Model', y=col, data=df, palette=pal, ax=ax)
        ax.set_title(title)
        if ylim:
            ax.set_ylim(0, ylim)
        for p in ax.patches:
            ax.annotate(
                f'{p.get_height():.4f}',
                (p.get_x() + p.get_width() / 2, p.get_height()),
                ha='center', va='bottom', fontsize=9,
            )

    plt.tight_layout()
    if save:
        plt.savefig('model_comparison.png', dpi=300, bbox_inches='tight')
        print("  Saved: model_comparison.png")
    if IN_COLAB:
        plt.show()
    else:
        plt.close(fig)

    df.to_csv('model_comparison_results.csv', index=False)
    print('✅ Comparison complete.')


def compare_models(data_path):
    """
    Run distilbert, distilbert-lora, distilbert-rag; then visualize,
    break-even analysis, and chunk-size ablation.
    """
    all_results = []

    for model_type in ['distilbert', 'distilbert-lora', 'distilbert-rag']:
        try:
            sys = QASystem(model_type)
            res = sys.run_experiment(data_path)
            if res:
                all_results.append(res)
        except Exception as e:
            print(f'\n❌ {model_type} failed: {e}')
            traceback.print_exc()

    if all_results:
        df = pd.DataFrame(all_results)
        df.to_csv('model_comparison_results.csv', index=False)
        visualize_comparison(all_results)
        break_even_analysis(df)

        print('\nRunning chunk-size ablation on fresh data load...')
        sys0 = QASystem('distilbert-rag')
        train_data, val_data = sys0.load_and_sample_data(data_path)
        chunk_size_ablation(train_data, val_data)

    return all_results


if __name__ == '__main__':
    DATA_PATH = '/kaggle/input/datasets/ytarunjitmeitei/qa-squad/adv_qa.json'
    compare_models(DATA_PATH)

In [ ]:
print("Hello")

In [ ]:
import importlib
import inspect
import shutil
import subprocess
import sys


def _version_tuple(version_str):
    parts = []
    for piece in str(version_str).split('.'):
        try:
            parts.append(int(piece))
        except ValueError:
            break
    return tuple(parts)


def ensure_torchao():
    try:
        import torchao
        if _version_tuple(torchao.__version__) < (0, 16, 0):
            raise ImportError(f'torchao {torchao.__version__} is too old')
        return
    except ImportError:
        pass
    print('Upgrading torchao for PEFT / LoRA compatibility...')
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'torchao>=0.16.0',
    ])


def ensure_hf_stack():
    try:
        from huggingface_hub import split_torch_state_dict_into_shards  # noqa: F401
        ensure_torchao()
        return
    except ImportError:
        pass

    print('Upgrading huggingface_hub / transformers...')
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
        'huggingface_hub>=0.26.2,<1.0',
        'transformers>=4.46.0',
        'accelerate>=0.33.0,<2.0',
        'safetensors>=0.4.0',
        'tokenizers>=0.20.0',
        'peft>=0.14.0',
        'sentence-transformers>=3.0.0,<4.0',
        'codecarbon>=2.3.0,<3.0',
        'scikit-learn>=1.3.0',
        'seaborn>=0.13.0',
        'psutil>=5.9.0',
        'faiss-cpu>=1.7.4',
        'torchao>=0.16.0',
    ])
    import huggingface_hub
    importlib.reload(huggingface_hub)
    ensure_torchao()


ensure_hf_stack()

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

import json
import os
import random
import time
import threading
import traceback

import numpy as np
import pandas as pd
import torch
import psutil
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from sklearn.model_selection import train_test_split
import faiss

import huggingface_hub
import codecarbon
from codecarbon import EmissionsTracker

from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    TrainingArguments,
    Trainer,
    TrainerCallback,
)
from peft import LoraConfig, get_peft_model, TaskType
from sentence_transformers import SentenceTransformer

ensure_torchao()

# =============================================================================
# Model IDs
# =============================================================================
T5_BASE_MODEL   = 't5-small'
T5_FT_MODEL     = 't5-small'  # warm-start for FT / LoRA
T5_RAG_READER   = 't5-small'  # pretrained reader (no FT in RAG)

MODEL_TYPES = ('t5-small', 't5-small-lora', 't5-small-rag')

# Fixed display order / colors for 3-panel plots
MODEL_DISPLAY = {
    't5-small'      : 'T5-small',
    't5-small-lora' : 'T5-small-LoRA',
    't5-small-rag'  : 'T5-small-RAG',
}
MODEL_COLORS = {
    'T5-small'      : '#4C72B0',
    'T5-small-LoRA' : '#DD8452',
    'T5-small-RAG'  : '#55A868',
}
RESULTS_CSV = 'model_comparison_results.csv'

# =============================================================================
# Reproducibility & experiment config
# =============================================================================
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# --- Tuned for higher F1 / EM ------------------------------------------------
NUM_EPOCHS                  = 3
BATCH_SIZE                  = 8
GRADIENT_ACCUMULATION_STEPS = 2
LEARNING_RATE               = 5e-5   # T5 QA fine-tuning (was 3e-4)
LORA_LEARNING_RATE          = 5e-5
SAMPLE_FRACTION             = 0.50
MAX_INPUT_LENGTH            = 384
MAX_ANSWER_LENGTH           = 64    # SQuAD answers are short
EVAL_SAMPLE_SIZE            = 50
RAG_TOP_K                   = 3
NUM_BEAMS                   = 4     # beam search at inference

CHUNK_CONFIG = {
    'chunk_size'   : 800,
    'chunk_overlap': 200,
    'strategy'     : 'sliding_window_character',
}

CC_CONFIG = {
    'country_iso_code'  : 'IND',
    'tracking_mode'     : 'machine',
    'log_level'         : 'warning',
    'save_to_file'      : True,
    'output_dir'        : 'emissions_data',
    'measure_power_secs': 10,
}

os.makedirs('emissions_data', exist_ok=True)
os.makedirs('results_comparison', exist_ok=True)

print('\n' + '=' * 65)
print('EXPERIMENT CONFIGURATION  (paper section 4.2)')
print('=' * 65)
print(f'  huggingface_hub      : {huggingface_hub.__version__}')
print(f'  transformers         : {__import__("transformers").__version__}')
print(f'  codecarbon version   : {codecarbon.__version__}')
print(f'  country_iso_code     : {CC_CONFIG["country_iso_code"]}')
print(f'  carbon_intensity     : 708 gCO2eq/kWh  (India CEA 2023)')
print(f'  device               : {DEVICE}')
if torch.cuda.is_available():
    print(f'  GPU                  : {torch.cuda.get_device_name(0)}')
print(f'  SAMPLE_FRACTION      : {SAMPLE_FRACTION}')
print(f'  NUM_EPOCHS           : {NUM_EPOCHS}')
print(f'  BATCH_SIZE           : {BATCH_SIZE}  (×{GRADIENT_ACCUMULATION_STEPS} accum)')
print(f'  LEARNING_RATE        : {LEARNING_RATE}  (LoRA: {LORA_LEARNING_RATE})')
print(f'  MAX_INPUT_LENGTH     : {MAX_INPUT_LENGTH}')
print(f'  MAX_ANSWER_LENGTH    : {MAX_ANSWER_LENGTH}')
print(f'  NUM_BEAMS (infer)    : {NUM_BEAMS}')
print(f'  EVAL_SAMPLE_SIZE     : {EVAL_SAMPLE_SIZE}')
print(f'  random_seed          : {RANDOM_SEED}')
print(f'  FT / LoRA checkpoint : {T5_FT_MODEL}')
print(f'  RAG_retriever        : sentence-transformers/all-MiniLM-L6-v2')
print(f'  RAG_reader           : {T5_RAG_READER}')
print(f'  RAG_knowledge_base   : AdvQA training split (same-dataset)')
print(f'  RAG_top_k            : {RAG_TOP_K}')
print(f'  RAG_search           : FAISS IndexFlatIP (cosine, L2-normalized)')
print('=' * 65 + '\n')


def format_t5_input(question, context):
    """Standard T5 SQuAD-style prefix."""
    return f"question: {question} context: {context}"


def build_faiss_index(embeddings):
    vectors = np.ascontiguousarray(np.asarray(embeddings, dtype=np.float32))
    if vectors.ndim != 2 or vectors.shape[0] == 0:
        raise ValueError('embeddings must be a non-empty 2-D array')
    faiss.normalize_L2(vectors)
    index = faiss.IndexFlatIP(vectors.shape[1])
    index.add(vectors)
    return index


def faiss_search(index, query_embeddings, top_k=RAG_TOP_K):
    queries = np.ascontiguousarray(np.asarray(query_embeddings, dtype=np.float32))
    if queries.ndim == 1:
        queries = queries.reshape(1, -1)
    faiss.normalize_L2(queries)
    k = min(int(top_k), index.ntotal)
    return index.search(queries, k)


def make_training_arguments(model_type, learning_rate=None):
    out_dir = f'./results_comparison/{model_type}'
    if os.path.isdir(out_dir):
        shutil.rmtree(out_dir, ignore_errors=True)
    os.makedirs(out_dir, exist_ok=True)

    lr = learning_rate if learning_rate is not None else LEARNING_RATE

    desired = {
        'output_dir'                  : out_dir,
        'overwrite_output_dir'        : True,
        'num_train_epochs'            : NUM_EPOCHS,
        'per_device_train_batch_size' : BATCH_SIZE,
        'per_device_eval_batch_size'  : BATCH_SIZE,
        'gradient_accumulation_steps' : GRADIENT_ACCUMULATION_STEPS,
        'learning_rate'               : lr,
        'weight_decay'                : 0.01,
        'warmup_ratio'                : 0.1,
        'warmup_steps'                : 100,
        'logging_dir'                 : './logs',
        'logging_steps'               : 50,
        'eval_strategy'               : 'epoch',
        'evaluation_strategy'         : 'epoch',
        'save_strategy'               : 'epoch',
        'load_best_model_at_end'      : True,
        'metric_for_best_model'       : 'eval_loss',
        'greater_is_better'           : False,
        'save_total_limit'            : 2,
        'label_smoothing_factor'      : 0.1,
        'report_to'                   : 'none',
        'disable_tqdm'                : False,
        'predict_with_generate'       : False,
    }
    if not torch.cuda.is_available():
        desired['use_cpu'] = True
        desired['no_cuda'] = True

    sig = inspect.signature(TrainingArguments.__init__)
    valid = set(sig.parameters) - {'self'}
    ta_kw = {k: v for k, v in desired.items() if k in valid}
    dropped = sorted(set(desired) - set(ta_kw))
    if dropped:
        print(f'  TrainingArguments: skipped unsupported keys: {dropped}')
    return TrainingArguments(**ta_kw)


def safe_emissions_tracker(output_file, **kwargs):
    save_to_file = kwargs.pop('save_to_file', CC_CONFIG['save_to_file'])
    measure_power_secs = kwargs.pop('measure_power_secs', CC_CONFIG['measure_power_secs'])
    common = dict(
        output_file=output_file,
        tracking_mode=CC_CONFIG['tracking_mode'],
        log_level=CC_CONFIG['log_level'],
        save_to_file=save_to_file,
        output_dir=CC_CONFIG['output_dir'],
        measure_power_secs=measure_power_secs,
        **kwargs,
    )
    try:
        return EmissionsTracker(country_iso_code=CC_CONFIG['country_iso_code'], **common)
    except TypeError:
        return EmissionsTracker(**common)


class TextChunker:
    def __init__(
        self,
        chunk_size    = CHUNK_CONFIG['chunk_size'],
        chunk_overlap = CHUNK_CONFIG['chunk_overlap'],
    ):
        assert chunk_overlap < chunk_size
        self.chunk_size    = chunk_size
        self.chunk_overlap = chunk_overlap
        self.step          = chunk_size - chunk_overlap

    def chunk_text(self, text, source_idx=None):
        chunks, text, start = [], text.strip(), 0
        while start < len(text):
            end   = min(start + self.chunk_size, len(text))
            chunk = text[start:end].strip()
            if len(chunk) > 30:
                chunks.append({
                    'text': chunk, 'char_start': start, 'char_end': end,
                    'source_idx': source_idx, 'chunk_id': len(chunks),
                })
            if end == len(text):
                break
            start += self.step
        return chunks

    def chunk_dataset(self, train_data):
        all_chunks = []
        for idx, example in enumerate(train_data):
            chunks = self.chunk_text(example['context'], source_idx=idx)
            for chunk in chunks:
                chunk['gold_answer'] = example['answers']['text'][0]
                chunk['question']    = example['question']
            all_chunks.extend(chunks)
        return all_chunks

    def print_stats(self, all_chunks, train_data):
        lens = [len(c['text']) for c in all_chunks]
        print('\n  Chunking statistics (paper section 3.3):')
        print(f'    original contexts  : {len(train_data)}')
        print(f'    total chunks       : {len(all_chunks)}')
        print(f'    avg chunks/context : {len(all_chunks)/len(train_data):.1f}')
        print(f'    avg chunk length   : {np.mean(lens):.0f} chars')
        print(f'    min / max length   : {min(lens)} / {max(lens)} chars')


class T5QADataset(torch.utils.data.Dataset):
    """Seq2seq dataset: source = question+context, target = answer text."""

    def __init__(self, tokenizer, data):
        self.tokenizer  = tokenizer
        self.data       = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        try:
            item   = self.data[idx]
            source = format_t5_input(item['question'], item['context'])
            target = item['answers']['text'][0]

            model_inputs = self.tokenizer(
                source,
                max_length=MAX_INPUT_LENGTH,
                truncation=True,
                padding='max_length',
            )
            labels = self.tokenizer(
                text_target=target,
                max_length=MAX_ANSWER_LENGTH,
                truncation=True,
                padding='max_length',
            )
            label_ids = [
                (tid if tid != self.tokenizer.pad_token_id else -100)
                for tid in labels['input_ids']
            ]
            model_inputs['labels'] = label_ids
            return model_inputs

        except Exception as e:
            print(f"T5QADataset error at idx {idx}: {e}")
            model_inputs = self.tokenizer(
                format_t5_input('dummy', 'context'),
                max_length=MAX_INPUT_LENGTH,
                truncation=True,
                padding='max_length',
            )
            labels = self.tokenizer(
                text_target='answer',
                max_length=MAX_ANSWER_LENGTH,
                truncation=True,
                padding='max_length',
            )
            label_ids = [
                (tid if tid != self.tokenizer.pad_token_id else -100)
                for tid in labels['input_ids']
            ]
            model_inputs['labels'] = label_ids
            return model_inputs


class CustomCallback(TrainerCallback):
    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step % args.logging_steps == 0:
            loss = (state.log_history[-1].get('loss', 'N/A')
                    if state.log_history else 'N/A')
            print(f"  Step {state.global_step}/{state.max_steps} — loss: {loss}")


class QASystem:
    """
    model_type: 't5-small' | 't5-small-lora' | 't5-small-rag'
    """

    def __init__(self, model_type='t5-small'):
        if model_type not in MODEL_TYPES:
            raise ValueError(f'Unknown model_type: {model_type}')
        self.model_type = model_type

    def _read_power(self):
        pw = {'cpu_power': 0.0, 'ram_power': 0.0, 'gpu_power': 0.0}
        try:
            pct  = psutil.cpu_percent(interval=0.1)
            freq = (psutil.cpu_freq().current / 1000 if psutil.cpu_freq() else 2.5)
            pw['cpu_power'] = (pct / 100) * psutil.cpu_count() * (15 + 10 * (freq - 2.5))
            pw['ram_power'] = (psutil.virtual_memory().percent / 100) * 5.0
            if torch.cuda.is_available():
                try:
                    pw['gpu_power'] = torch.cuda.power_draw(torch.cuda.current_device()) / 1000
                except Exception:
                    pw['gpu_power'] = 75.0
        except Exception as e:
            print(f"Power read error: {e}")
        return pw

    def _power_thread(self, readings, stop_evt):
        while not stop_evt.is_set():
            readings.append(self._read_power())
            time.sleep(1)

    def _summarise_power(self, readings, prefix):
        if not readings:
            return {f'{prefix}_cpu': 0, f'{prefix}_ram': 0,
                    f'{prefix}_gpu': 0, f'{prefix}_total': 0}
        df = pd.DataFrame(readings)
        return {
            f'{prefix}_cpu'  : float(df['cpu_power'].mean()),
            f'{prefix}_ram'  : float(df['ram_power'].mean()),
            f'{prefix}_gpu'  : float(df['gpu_power'].mean()),
            f'{prefix}_total': float(df[['cpu_power', 'ram_power', 'gpu_power']].sum(axis=1).mean()),
        }

    def load_and_sample_data(self, file_path, fraction=SAMPLE_FRACTION):
        print(f"\nLoading {fraction*100:.0f}% sample from {file_path}...")
        if not os.path.exists(file_path):
            raise FileNotFoundError(f"Dataset not found: {file_path}")

        with open(file_path, 'r') as f:
            first = f.read(1); f.seek(0)
            raw = (json.load(f) if first == '['
                   else [json.loads(l) for l in f if l.strip()])

        valid = []
        for ex in raw:
            try:
                if not all(k in ex for k in ['context', 'question', 'answers']):
                    continue
                ans = ex['answers']
                if not (isinstance(ans, dict)
                        and all(k in ans for k in ['text', 'answer_start'])
                        and ans['text'] and ans['answer_start']):
                    continue
                ctx, a_s, a_t = ex['context'], ans['answer_start'][0], ans['text'][0]
                a_e = a_s + len(a_t)
                if not (0 <= a_s < len(ctx) and a_e <= len(ctx) and ctx[a_s:a_e] == a_t):
                    continue
                valid.append(ex)
            except Exception:
                continue

        if not valid:
            raise ValueError("No valid QA pairs found in dataset.")

        n       = min(int(len(valid) * fraction), len(valid))
        sampled = random.sample(valid, n)
        train_data, val_data = train_test_split(
            sampled, test_size=0.2, random_state=RANDOM_SEED
        )
        print(f"✓ {len(valid)} valid → {n} sampled → "
              f"{len(train_data)} train / {len(val_data)} val")
        return train_data, val_data

    def initialize_model(self):
        print(f"\nLoading {self.model_type}...")
        try:
            if self.model_type == 't5-small':
                name = T5_FT_MODEL
                tok  = AutoTokenizer.from_pretrained(name)
                mdl  = AutoModelForSeq2SeqLM.from_pretrained(name).to(DEVICE)

            elif self.model_type == 't5-small-lora':
                ensure_torchao()
                name = T5_FT_MODEL
                tok  = AutoTokenizer.from_pretrained(name)
                mdl  = AutoModelForSeq2SeqLM.from_pretrained(name).to(DEVICE)
                cfg  = LoraConfig(
                    task_type      = TaskType.SEQ_2_SEQ_LM,
                    inference_mode = False,
                    r              = 8,
                    lora_alpha     = 32,
                    lora_dropout   = 0.1,
                    target_modules = ['q', 'v'],
                )
                mdl = get_peft_model(mdl, cfg)
                mdl.print_trainable_parameters()

            elif self.model_type == 't5-small-rag':
                name = T5_RAG_READER
                tok  = AutoTokenizer.from_pretrained(name)
                mdl  = AutoModelForSeq2SeqLM.from_pretrained(name).to(DEVICE)

            else:
                raise ValueError(f"Unknown model_type: {self.model_type}")

            print(f"✓ {self.model_type} loaded on {next(mdl.parameters()).device}")
            return {'model': mdl, 'tokenizer': tok}

        except Exception as e:
            print(f"❌ Model loading failed: {e}")
            if 'torchao' in str(e).lower():
                print("Fix: pip install -q 'torchao>=0.16.0' then restart runtime.")
            return None

    def train_t5_model(self, model_dict, train_data, val_data):
        """Full fine-tune or LoRA training for T5-small (seq2seq)."""
        tok      = model_dict['tokenizer']
        train_ds = T5QADataset(tok, train_data)
        val_ds   = T5QADataset(tok, val_data)
        lr = LORA_LEARNING_RATE if self.model_type == 't5-small-lora' else LEARNING_RATE
        args     = make_training_arguments(self.model_type, learning_rate=lr)
        collator = DataCollatorForSeq2Seq(
            tokenizer=tok,
            model=model_dict['model'],
            label_pad_token_id=-100,
        )

        trainer = Trainer(
            model           = model_dict['model'],
            args            = args,
            train_dataset   = train_ds,
            eval_dataset    = val_ds,
            data_collator   = collator,
            callbacks       = [CustomCallback()],
        )

        readings = []
        stop_evt = threading.Event()
        p_thread = threading.Thread(
            target=self._power_thread, args=(readings, stop_evt), daemon=True
        )
        tracker = safe_emissions_tracker(
            output_file=f'emissions_{self.model_type}_train.csv'
        )

        print(f"\n🔧 Training {self.model_type}...")
        emissions = 0.0
        try:
            p_thread.start()
            tracker.start()
            trainer.train()
            emissions = tracker.stop()
        except Exception:
            try:
                tracker.stop()
            except Exception:
                pass
            raise
        finally:
            stop_evt.set()
            p_thread.join()

        print(f"✓ Training emissions: {emissions:.8f} kg CO₂")
        return model_dict, emissions, self._summarise_power(readings, 'train')

    def train_rag_model(self, model_dict, train_data, val_data):
        """RAG Phase 1: chunk + embed + FAISS index."""
        print('\n📚 RAG — offline indexing (Phase 1)...')
        print(f'   retriever : all-MiniLM-L6-v2')
        print(f'   reader    : {T5_RAG_READER}  (top-{RAG_TOP_K} + best beam)')
        print(f'   KB source : AdvQA training split (same-dataset)')

        retriever = SentenceTransformer('all-MiniLM-L6-v2')
        chunker   = TextChunker()

        readings, stop_evt = [], threading.Event()
        p_thread = threading.Thread(
            target=self._power_thread, args=(readings, stop_evt), daemon=True
        )
        tracker = safe_emissions_tracker(
            output_file='emissions_t5-small-rag_indexing.csv'
        )

        t_start = time.time()
        all_chunks, emissions = [], 0.0

        try:
            p_thread.start()
            tracker.start()

            all_chunks = chunker.chunk_dataset(train_data)
            chunker.print_stats(all_chunks, train_data)

            chunk_texts  = [c['text'] for c in all_chunks]
            ENCODE_BATCH = 64
            all_embeddings = []
            print(f'\n   Encoding {len(all_chunks)} chunks...')
            for i in tqdm(range(0, len(chunk_texts), ENCODE_BATCH), desc='   Encoding'):
                batch = chunk_texts[i: i + ENCODE_BATCH]
                all_embeddings.append(
                    retriever.encode(batch, convert_to_numpy=True, show_progress_bar=False)
                )
            all_embeddings = np.vstack(all_embeddings).astype(np.float32)
            emissions = tracker.stop()

        except Exception:
            try:
                tracker.stop()
            except Exception:
                pass
            raise
        finally:
            stop_evt.set()
            p_thread.join()

        elapsed = time.time() - t_start
        pd.DataFrame([{
            'num_source_contexts'  : len(train_data),
            'num_chunks'           : len(all_chunks),
            'chunk_size_chars'     : CHUNK_CONFIG['chunk_size'],
            'chunk_overlap_chars'  : CHUNK_CONFIG['chunk_overlap'],
            'embedding_model'      : 'all-MiniLM-L6-v2',
            'reader_model'         : T5_RAG_READER,
            'storage'              : 'FAISS IndexFlatIP (in-memory)',
            'search_method'        : 'FAISS inner product (L2-normalized cosine)',
            'top_k'                : RAG_TOP_K,
            'indexing_time_sec'    : round(elapsed, 2),
            'indexing_emissions_kg': emissions,
            'country_iso_code'     : CC_CONFIG['country_iso_code'],
        }]).to_csv('emissions_data/rag_indexing_metadata.csv', index=False)

        print(f'\n✓ Indexing: {emissions:.8f} kg CO₂  in {elapsed:.1f}s')
        print(f'   Building FAISS index ({all_embeddings.shape[0]} vectors)...')
        faiss_index = build_faiss_index(all_embeddings)

        model_dict['retriever']      = retriever
        model_dict['all_chunks']     = all_chunks
        model_dict['all_embeddings'] = all_embeddings
        model_dict['faiss_index']    = faiss_index
        return model_dict, emissions, self._summarise_power(readings, 'train')

    def _retrieve_chunks(self, model_dict, question, top_k=RAG_TOP_K):
        q_emb = model_dict['retriever'].encode(
            question, convert_to_numpy=True, show_progress_bar=False,
        )
        scores, indices = faiss_search(model_dict['faiss_index'], q_emb, top_k=top_k)
        results = []
        for rank in range(indices.shape[1]):
            idx = int(indices[0][rank])
            results.append((
                idx,
                float(scores[0][rank]),
                model_dict['all_chunks'][idx]['text'],
            ))
        return results

    def _generate_answer(self, model, tok, question, context):
        """T5 reader: beam search generation; returns (answer, beam_score)."""
        source = format_t5_input(question, context)
        inputs = tok(
            source,
            return_tensors='pt',
            truncation=True,
            max_length=MAX_INPUT_LENGTH,
        ).to(DEVICE)

        gen_kw = dict(
            max_new_tokens=MAX_ANSWER_LENGTH,
            num_beams=NUM_BEAMS,
            early_stopping=True,
            no_repeat_ngram_size=2,
            length_penalty=1.0,
        )

        with torch.no_grad():
            try:
                out = model.generate(
                    **inputs,
                    return_dict_in_generate=True,
                    output_scores=True,
                    **gen_kw,
                )
                seq   = out.sequences[0]
                score = float(out.sequences_scores[0].item()) if out.sequences_scores is not None else 0.0
            except TypeError:
                seq = model.generate(**inputs, **gen_kw)[0]
                score = 0.0

        answer = tok.decode(seq, skip_special_tokens=True).strip()
        return answer, score

    def _predict_rag_answer(self, model, tok, model_dict, question):
        """Generate on top-k chunks; keep answer with best beam score."""
        chunks = self._retrieve_chunks(model_dict, question, top_k=RAG_TOP_K)
        best_answer, best_score = '', float('-inf')
        best_sim = chunks[0][1] if chunks else 0.0

        for _, sim, context in chunks:
            answer, beam_score = self._generate_answer(model, tok, question, context)
            if beam_score > best_score and answer:
                best_score = beam_score
                best_answer = answer
                best_sim = sim

        if not best_answer and chunks:
            best_answer, _ = self._generate_answer(model, tok, question, chunks[0][2])
        return best_answer, best_sim, chunks

    def evaluate_model(self, model_dict, val_data):
        eval_emissions, f1_scores, exact_matches = [], [], []
        power_data, retrieval_hits = [], []

        model = model_dict['model']
        tok   = model_dict['tokenizer']
        model.eval()

        n_eval = min(len(val_data), EVAL_SAMPLE_SIZE)
        print(f"\n🔍 Evaluating {self.model_type} on {n_eval} examples...")

        for idx, qa_pair in enumerate(tqdm(val_data[:n_eval], desc='Eval')):
            readings, stop_evt = [], threading.Event()
            p_thread = threading.Thread(
                target=self._power_thread, args=(readings, stop_evt), daemon=True
            )
            tracker = safe_emissions_tracker(
                output_file=f'emissions_{self.model_type}_eval.csv',
                save_to_file=False,
                measure_power_secs=1,
            )

            p_thread.start()
            emissions = 0.0
            try:
                tracker.start()

                if self.model_type == 't5-small-rag':
                    pred_answer, best_sim, chunks = self._predict_rag_answer(
                        model, tok, model_dict, qa_pair['question']
                    )
                    gold_ans = qa_pair['answers']['text'][0]
                    ans_in_any = any(
                        gold_ans.lower() in c[2].lower() for c in chunks
                    )
                    retrieval_hits.append(int(ans_in_any))

                    if idx < 5:
                        print(f"\n  [RAG debug #{idx}] top_k={len(chunks)} "
                              f"ans_in_topk={ans_in_any} sim={best_sim:.4f}")
                else:
                    pred_answer, _ = self._generate_answer(
                        model, tok, qa_pair['question'], qa_pair['context']
                    )
                em, f1 = self.calculate_metrics(pred_answer, qa_pair['answers']['text'])
                exact_matches.append(em)
                f1_scores.append(f1)

            except Exception as e:
                print(f"\n⚠️ Eval error #{idx}: {str(e)[:120]}")
                exact_matches.append(0)
                f1_scores.append(0)
                if self.model_type == 't5-small-rag':
                    retrieval_hits.append(0)

            finally:
                try:
                    emissions = tracker.stop()
                except Exception:
                    emissions = 0.0
                stop_evt.set()
                p_thread.join()
                eval_emissions.append(emissions)
                power_data.append(self._summarise_power(readings, 'eval'))

        if retrieval_hits:
            retr_acc = float(np.mean(retrieval_hits))
            print(f'\n📊 Retrieval accuracy (answer in chunk): {retr_acc*100:.1f}%')
            pd.DataFrame([{
                'retrieval_accuracy': retr_acc,
                'n_correct_retrieval': sum(retrieval_hits),
                'n_total': len(retrieval_hits),
                'reader_model': T5_RAG_READER,
                'search_backend': 'FAISS IndexFlatIP',
            }]).to_csv('emissions_data/rag_retrieval_diagnostic.csv', index=False)

        avg_ep = {
            'cpu'  : float(np.mean([p['eval_cpu']   for p in power_data])),
            'ram'  : float(np.mean([p['eval_ram']   for p in power_data])),
            'gpu'  : float(np.mean([p['eval_gpu']   for p in power_data])),
            'total': float(np.mean([p['eval_total'] for p in power_data])),
        }
        return eval_emissions, f1_scores, exact_matches, avg_ep

    @staticmethod
    def calculate_metrics(pred, true_answers):
        def norm(t):
            t = str(t).lower()
            t = ''.join(c for c in t if c.isalnum() or c.isspace())
            return ' '.join(t.split())

        p  = norm(pred)
        ts = [norm(a) for a in true_answers]
        em = int(any(p == t for t in ts))

        p_tok, best = p.split(), 0.0
        for t in ts:
            t_tok = t.split()
            if not p_tok or not t_tok:
                continue
            common = set(p_tok) & set(t_tok)
            prec   = len(common) / len(p_tok)
            rec    = len(common) / len(t_tok)
            f1     = (2 * prec * rec / (prec + rec)) if (prec + rec) > 0 else 0.0
            best   = max(best, f1)
        return em, best

    def diagnose_rag_performance(self, model_dict, val_data):
        if self.model_type != 't5-small-rag':
            return

        print('\n🔬 RAG performance diagnostic (paper section 5.3)...')
        model, tok = model_dict['model'], model_dict['tokenizer']
        model.eval()

        correct_f1, wrong_f1 = [], []
        correct_em, wrong_em = [], []
        hits = []

        for qa in tqdm(val_data[:EVAL_SAMPLE_SIZE], desc='  Diagnostic'):
            try:
                pred, _, chunks = self._predict_rag_answer(
                    model, tok, model_dict, qa['question']
                )
                gold = qa['answers']['text'][0]
                hit  = int(any(gold.lower() in c[2].lower() for c in chunks))
                hits.append(hit)

                em, f1 = self.calculate_metrics(pred, qa['answers']['text'])

                if hit:
                    correct_f1.append(f1); correct_em.append(em)
                else:
                    wrong_f1.append(f1);   wrong_em.append(em)
            except Exception:
                hits.append(0)
                wrong_f1.append(0); wrong_em.append(0)

        diag = {
            'retrieval_accuracy'  : float(np.mean(hits)),
            'overall_f1'          : float(np.mean(correct_f1 + wrong_f1)) if hits else 0.0,
            'f1_given_correct_ctx': float(np.mean(correct_f1)) if correct_f1 else 0.0,
            'f1_given_wrong_ctx'  : float(np.mean(wrong_f1))   if wrong_f1   else 0.0,
            'overall_em'          : float(np.mean(correct_em + wrong_em)) if hits else 0.0,
            'em_given_correct_ctx': float(np.mean(correct_em)) if correct_em else 0.0,
            'em_given_wrong_ctx'  : float(np.mean(wrong_em))   if wrong_em   else 0.0,
            'n_correct_retrieval' : int(sum(hits)),
            'n_wrong_retrieval'   : int(len(hits) - sum(hits)),
        }

        print('\n  Diagnostic results:')
        for k, v in diag.items():
            print(f'    {k:<30} : {v:.4f}' if isinstance(v, float) else f'    {k:<30} : {v}')

        pd.DataFrame([diag]).to_csv(
            'emissions_data/rag_performance_diagnostic.csv', index=False
        )
        return diag

    def run_experiment(self, data_path):
        print(f'\n{"="*60}')
        print(f'🚀 {self.model_type.upper()} experiment')
        print(f'   device={DEVICE}  epochs={NUM_EPOCHS}  batch={BATCH_SIZE}')

        train_data, val_data = self.load_and_sample_data(data_path)
        model_dict           = self.initialize_model()
        if not model_dict:
            return None

        if self.model_type == 't5-small-rag':
            model_dict, t_em, t_pw = self.train_rag_model(model_dict, train_data, val_data)
            self.diagnose_rag_performance(model_dict, val_data)
        else:
            model_dict, t_em, t_pw = self.train_t5_model(model_dict, train_data, val_data)

        e_ems, f1s, ems, e_pw = self.evaluate_model(model_dict, val_data)

        is_rag = self.model_type == 't5-small-rag'
        results = {
            'Model'              : self.model_type,
            'Train_CO2_kg'       : t_em,
            'Train_phase'        : 'indexing' if is_rag else 'fine-tuning',
            'Eval_CO2_kg_mean'   : float(np.mean(e_ems)),
            'Eval_CO2_kg_std'    : float(np.std(e_ems)),
            'Avg_F1'             : float(np.mean(f1s)),
            'Std_F1'             : float(np.std(f1s)),
            'Avg_EM'             : float(np.mean(ems)),
            'Std_EM'             : float(np.std(ems)),
            'Train_CPU_W'        : t_pw['train_cpu'],
            'Train_RAM_W'        : t_pw['train_ram'],
            'Train_GPU_W'        : t_pw['train_gpu'],
            'Train_Total_W'      : t_pw['train_total'],
            'Eval_CPU_W'         : e_pw['cpu'],
            'Eval_RAM_W'         : e_pw['ram'],
            'Eval_GPU_W'         : e_pw['gpu'],
            'Eval_Total_W'       : e_pw['total'],
            'Num_Train'          : len(train_data),
            'Num_Eval'           : min(len(val_data), EVAL_SAMPLE_SIZE),
            'Batch_Size'         : BATCH_SIZE,
            'Epochs'             : NUM_EPOCHS,
            'Chunk_Size_Chars'   : CHUNK_CONFIG['chunk_size'] if is_rag else 'N/A',
            'Chunk_Overlap_Chars': CHUNK_CONFIG['chunk_overlap'] if is_rag else 'N/A',
        }

        print(f'\n🎯 {self.model_type} results:')
        print(pd.DataFrame([results])[
            ['Model', 'Train_CO2_kg', 'Eval_CO2_kg_mean', 'Avg_F1', 'Avg_EM']
        ].to_string(index=False))
        return results


def chunk_size_ablation(train_data, val_data, save=True):
    print('\n📐 Chunk-size ablation...')
    retriever = SentenceTransformer('all-MiniLM-L6-v2')
    configs = [
        {'chunk_size': 400,  'chunk_overlap': 100},
        {'chunk_size': 512,  'chunk_overlap': 128},   # current default
        {'chunk_size': 800,  'chunk_overlap': 200},
    ]
    rows = []
    for cfg in configs:
        chunker    = TextChunker(cfg['chunk_size'], cfg['chunk_overlap'])
        all_chunks = chunker.chunk_dataset(train_data)
        embs       = retriever.encode(
            [c['text'] for c in all_chunks],
            convert_to_numpy=True, batch_size=64, show_progress_bar=False,
        ).astype(np.float32)
        faiss_index = build_faiss_index(embs)

        hits = []
        for qa in val_data[:EVAL_SAMPLE_SIZE]:
            q_emb = retriever.encode(
                [qa['question']], convert_to_numpy=True, show_progress_bar=False
            )
            _, indices = faiss_search(faiss_index, q_emb, top_k=1)
            chunk = all_chunks[int(indices[0][0])]['text']
            gold  = qa['answers']['text'][0]
            hits.append(int(gold.lower() in chunk.lower()))

        row = {
            'chunk_size': cfg['chunk_size'],
            'chunk_overlap': cfg['chunk_overlap'],
            'num_chunks': len(all_chunks),
            'retrieval_acc': round(float(np.mean(hits)), 4),
        }
        rows.append(row)
        print(f"  chunk={cfg['chunk_size']:5d}  acc={row['retrieval_acc']:.4f}")

    df = pd.DataFrame(rows)
    if save:
        df.to_csv('emissions_data/chunk_ablation.csv', index=False)
    print(df.to_string(index=False))
    return df


def break_even_analysis(results_df, save=True):
    print('\n📈 Break-even analysis...')

    ft   = results_df[results_df['Model'] == 't5-small']
    lora = results_df[results_df['Model'] == 't5-small-lora']
    rag  = results_df[results_df['Model'] == 't5-small-rag']

    if ft.empty or lora.empty or rag.empty:
        print('Need all 3 model results — skipping break-even analysis.')
        return None, None

    E_train_ft   = float(ft['Train_CO2_kg'].values[0])
    E_train_lora = float(lora['Train_CO2_kg'].values[0])
    E_index_rag  = float(rag['Train_CO2_kg'].values[0])
    E_infer_ft   = float(ft['Eval_CO2_kg_mean'].values[0])
    E_infer_lora = float(lora['Eval_CO2_kg_mean'].values[0])
    E_infer_rag  = float(rag['Eval_CO2_kg_mean'].values[0])

    Q      = np.arange(0, 10001, 100)
    E_ft   = (E_train_ft   + Q * E_infer_ft)   * 1000
    E_lora = (E_train_lora + Q * E_infer_lora) * 1000
    E_rag  = (E_index_rag  + Q * E_infer_rag)  * 1000

    def q_star(E_fa, E_va, E_fb, E_vb):
        d = E_vb - E_va
        if abs(d) < 1e-12:
            return None
        q = (E_fa - E_fb) / d
        return float(q) if q > 0 else None

    q_ft_rag   = q_star(E_train_ft,   E_infer_ft,   E_index_rag, E_infer_rag)
    q_lora_rag = q_star(E_train_lora, E_infer_lora, E_index_rag, E_infer_rag)

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(Q, E_ft,   label=MODEL_DISPLAY['t5-small'],      color=MODEL_COLORS['T5-small'],      lw=2)
    ax.plot(Q, E_lora, label=MODEL_DISPLAY['t5-small-lora'],   color=MODEL_COLORS['T5-small-LoRA'], lw=2)
    ax.plot(Q, E_rag,  label=MODEL_DISPLAY['t5-small-rag'],   color=MODEL_COLORS['T5-small-RAG'],  lw=2, ls='--')

    for q, col, lbl in [(q_ft_rag, 'steelblue', 'FT–RAG'), (q_lora_rag, 'darkorange', 'LoRA–RAG')]:
        if q and q <= 10000:
            ax.axvline(q, color=col, ls=':', alpha=0.7, label=f'{lbl} Q*≈{int(q):,}')

    ax.set_xlabel('Number of queries (Q)')
    ax.set_ylabel('Cumulative CO₂ (g CO₂eq)')
    ax.set_title('Break-even: T5-small adaptation vs query volume')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()

    if save:
        plt.savefig('breakeven_analysis.png', dpi=300, bbox_inches='tight')
        print("  Saved: breakeven_analysis.png")
    if IN_COLAB:
        plt.show()
    else:
        plt.close(fig)

    pd.DataFrame([{
        'E_train_ft_kg': E_train_ft, 'E_train_lora_kg': E_train_lora,
        'E_index_rag_kg': E_index_rag,
        'E_infer_ft_kg': E_infer_ft, 'E_infer_lora_kg': E_infer_lora,
        'E_infer_rag_kg': E_infer_rag,
        'Q_star_ft_rag': q_ft_rag, 'Q_star_lora_rag': q_lora_rag,
    }]).to_csv('emissions_data/breakeven_results.csv', index=False)

    return q_ft_rag, q_lora_rag


def clear_gpu_cache():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def merge_results(results=None, csv_path=RESULTS_CSV):
    """Merge in-memory results with CSV; one row per model (latest wins)."""
    frames = []
    if results is not None:
        frames.append(pd.DataFrame(results) if isinstance(results, list) else results.copy())
    if os.path.exists(csv_path):
        frames.append(pd.read_csv(csv_path))
    if not frames:
        return pd.DataFrame()
    df = pd.concat(frames, ignore_index=True)
    df = df[df['Model'].isin(MODEL_TYPES)].drop_duplicates(subset=['Model'], keep='last')
    df['_order'] = df['Model'].map({m: i for i, m in enumerate(MODEL_TYPES)})
    return df.sort_values('_order').drop(columns='_order').reset_index(drop=True)


def save_results(df, csv_path=RESULTS_CSV):
    df.to_csv(csv_path, index=False)
    return df


def visualize_comparison(results=None, save=True, csv_path=RESULTS_CSV):
    """
  Plot all three adaptations on shared axes:
  T5-small | T5-small-LoRA | T5-small-RAG

  Pass `results` and/or read existing rows from model_comparison_results.csv.
    """
    df = merge_results(results, csv_path)
    if df.empty:
        print('No results to plot. Run compare_models() or a single QASystem experiment first.')
        return df

    labels_all = [MODEL_DISPLAY[m] for m in MODEL_TYPES]
    missing    = [m for m in MODEL_TYPES if m not in set(df['Model'])]
    if missing:
        print('⚠️  Missing models (no bar plotted):',
              ', '.join(MODEL_DISPLAY[m] for m in missing))
        print('   Run: QASystem("t5-small-rag").run_experiment(DATA_PATH)  # example')

    fig, axes = plt.subplots(1, 3, figsize=(21, 6))
    fig.suptitle('T5-small adaptation comparison', fontsize=15, fontweight='bold')

    specs = [
        ('Train_CO2_kg', 'Training CO₂ (kg)', None),
        ('Avg_F1',       'Average F1',        1.0),
        ('Avg_EM',       'Average exact match', 1.0),
    ]

    x = np.arange(len(MODEL_TYPES))
    for ax, (col, title, ylim) in zip(axes, specs):
        heights = []
        for m in MODEL_TYPES:
            row = df[df['Model'] == m]
            heights.append(float(row[col].iloc[0]) if len(row) else np.nan)

        colors = [MODEL_COLORS[l] for l in labels_all]
        bars   = ax.bar(x, heights, color=colors, width=0.62, edgecolor='#333333', linewidth=0.6)
        ax.set_xticks(x)
        ax.set_xticklabels(labels_all, fontsize=11)
        ax.set_title(title, fontsize=12)
        ax.grid(axis='y', alpha=0.25)

        if ylim is not None:
            ax.set_ylim(0, ylim)
        else:
            valid = [h for h in heights if not np.isnan(h)]
            if valid:
                ax.set_ylim(0, max(valid) * 1.18)

        for bar, h in zip(bars, heights):
            if np.isnan(h):
                ax.text(
                    bar.get_x() + bar.get_width() / 2, 0.01,
                    'N/A', ha='center', va='bottom', fontsize=9, color='#888888',
                )
            else:
                ax.annotate(
                    f'{h:.4f}',
                    (bar.get_x() + bar.get_width() / 2, bar.get_height()),
                    ha='center', va='bottom', fontsize=9,
                )

    if len(df) == len(MODEL_TYPES):
        subtitle = 'All three adaptations included'
    else:
        subtitle = f'{len(df)}/{len(MODEL_TYPES)} adaptations in plot'
    fig.text(0.5, 0.01, subtitle, ha='center', fontsize=10, color='#555555')

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    if save:
        plt.savefig('model_comparison.png', dpi=300, bbox_inches='tight')
        print('  Saved: model_comparison.png')
    if IN_COLAB:
        plt.show()
    else:
        plt.close(fig)

    save_results(df, csv_path)
    print('✅ Comparison plot complete.')
    print(df[['Model', 'Train_CO2_kg', 'Avg_F1', 'Avg_EM']].to_string(index=False))
    return df


def plot_t5_comparison(csv_path=RESULTS_CSV, save=True):
    """Re-plot from saved CSV without re-running experiments."""
    return visualize_comparison(results=None, save=save, csv_path=csv_path)


def compare_models(data_path, plot=True, run_ablation=True):
    """Run t5-small, t5-small-lora, t5-small-rag; plot all three on one figure."""
    all_results = []

    for model_type in MODEL_TYPES:
        print(f'\n{"#"*60}\n  Pipeline: {MODEL_DISPLAY[model_type]}\n{"#"*60}')
        clear_gpu_cache()
        try:
            res = QASystem(model_type).run_experiment(data_path)
            if res:
                all_results.append(res)
                save_results(merge_results(all_results))
        except Exception as e:
            print(f'\n❌ {model_type} failed: {e}')
            traceback.print_exc()
        finally:
            clear_gpu_cache()

    df = merge_results(all_results)
    if df.empty:
        print('No experiments completed successfully.')
        return all_results

    if plot:
        visualize_comparison(df)

    if len(df) == len(MODEL_TYPES):
        break_even_analysis(df)
    else:
        print(f'\nBreak-even skipped: need 3 models, have {len(df)} '
              f'({list(df["Model"])}).')

    if run_ablation and 't5-small-rag' in set(df['Model']):
        print('\nRunning chunk-size ablation...')
        train_data, val_data = QASystem('t5-small-rag').load_and_sample_data(data_path)
        chunk_size_ablation(train_data, val_data)
    elif run_ablation:
        print('\nChunk ablation skipped (RAG not in results).')

    return all_results


if __name__ == '__main__':
    compare_models('/kaggle/input/datasets/ytarunjitmeitei/qa-squad/adv_qa.json')


In [ ]:
print("hello")